# ⚛️ TrackLab — User Guide: Workflows & Analysis Modes

Welcome to the **TrackLab User Guide**. This interactive notebook walks you through the physical modeling capabilities of TrackLab. You will learn how the physics engine works, how to use the different analysis modes, and how to run simulations both via the **Graphical User Interface (GUI)** and programmatically using **Python**.

### Table of Contents
1. **Section 1**: [Plotting and Comparing V(y) Etch-Rate Curves](#section1)
2. **Section 2**: [Setting the Simulation Configuration](#section2)
3. **Section 3**: [Single Track Simulation & Visualization](#section3)
4. **Section 4**: [Generating and Using Look-Up Tables (LUT)](#section4)
5. **Section 5**: [Processing Monte Carlo (FLUKA) Simulations](#section5)

In [ ]:
import sys
import os

# Ensure the TrackLab package root is accessible
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd

<a id="section1"></a>
## 📊 Section 1: Plotting and Comparing V(y) Etch-Rate Curves

The reduced etch-rate ratio $V(y) = V_T(y)/V_B$ describes the ratio of the track etch rate ($V_T$) to the bulk etch rate ($V_B$) along the particle trajectory, as a function of the residual range $y$.

### 🖥️ GUI Usage (Mode 1: V(y) Explorer)
1. Launch the GUI with `python run_gui.py`.
2. Navigate to the **1: V(y) Curve** tab.
3. Select the ion from the dropdown menu (e.g., `protons`, `C`, `Li`, etc.).
4. Enter the particle energy and click **Calculate** to see the curve overlaid with literature models.

### 💻 Programmatic Usage
Let's plot the $V(y)$ curves for Protons, Lithium, and Carbon using the TrackLab API:

In [ ]:
from tracklab.vt_utils import vt_function
from tracklab.vt_multiion import get_model

y = np.linspace(0.05, 40.0, 500)
vt_model = get_model()

# 1. Protons (Nikezic/Hermsdorf analytical model)
v_proton = [float(vt_function(yi)) for yi in y]

# 2. Lithium (Broken Power Law)
v_li = [float(vt_model.V(yi, ion='Li', energy=4.82, vb=1.73)) for yi in y]

# 3. Carbon (Broken Power Law)
v_c = [float(vt_model.V(yi, ion='C', energy=14.8, vb=1.73)) for yi in y]

# Plot the V(y) models
plt.figure(figsize=(9, 5.5))
plt.plot(y, v_proton, label="Protons (Nikezic/Hermsdorf)", lw=2)
plt.plot(y, v_li, label="Lithium (4.82 MeV, BPL)", lw=2)
plt.plot(y, v_c, label="Carbon (14.8 MeV, BPL)", lw=2)
plt.axhline(1, ls="--", color="gray", alpha=0.5)
plt.xlabel("Residual range y (µm)")
plt.ylabel("Reduced Etch-Rate Ratio V(y)")
plt.title("TrackLab V(y) Model Comparison")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<a id="section2"></a>
## ⚙️ Section 2: Setting the Simulation Configuration

To run track simulations, you need to define the physical environment parameters:
* **Bulk Etch Rate ($V_B$)**: The rate at which the chemical etchant dissolves the undamaged bulk detector material (typically in $\mu m/h$).
* **Etching Time ($t$)**: The duration of chemical etching (in hours).
* **Initial Energy ($E$)**: The initial kinetic energy of the particle (in MeV).
* **Incidence Angle ($\theta$)**: The angle of entry with respect to the detector surface (in degrees, where $90^\circ$ is normal incidence).

### 🖥️ GUI Configuration
In the GUI, these parameters are located in the **left-hand settings panel** under 'Ion Selection' and 'Physical Parameters'. They apply globally to the active tab.

### 💻 Programmatic Configuration
Let's see how to retrieve bulk etch rates and load the SRIM range-energy datasets programmatically:

In [ ]:
from tracklab.config import get_vb_for_ion
from tracklab.load_srim_data import load_srim_data

# Retrieve default bulk etch rates (VB) for different ions
for ion in ['protons', 'alpha', 'Li', 'C', 'O']:
    print(f"Default VB for {ion:8s}: {get_vb_for_ion(ion):.2f} µm/h")

# Load precomputed SRIM range-energy databases
range_tables, _ = load_srim_data()
print("\nSRIM database loaded. Available ions:", list(range_tables.keys()))

<a id="section3"></a>
## 🔍 Section 3: Single Track Simulation & Visualization

This mode allows you to simulate the development of a single particle track and analyze its resulting morphology.

### 🖥️ GUI Usage (Mode 2: Single Track)
1. Navigate to the **2: Single Track** tab.
2. Adjust parameters in the left panel (e.g., Proton, 1.5 MeV, 75° angle, $t = 2.83$ h).
3. Click **Calculate Track**.
4. The GUI displays:
   * **3D View**: An interactive 3D model of the track cavity.
   * **XY Microscope View**: A top-down optical simulation showing the track's contrast as seen under a transmission microscope.
   * **YZ and XZ Profiles**: Longitudinal cross-sections of the track walls.

### 💻 Programmatic Usage
Let's run a single track simulation programmatically and plot these exact four views:

In [ ]:
from tracklab.calculate_track_parameter import calculate_track_parameters
from tracklab.vt_utils import build_vrint_interpolator

# Configure track simulation
ion = 'protons'
energy = 1.5    # MeV
angle = 75.0    # degrees
vb = get_vb_for_ion(ion)      # 4.7 µm/h
time_etching = 2.83           # hours

# Load SRIM range interpolator
range_interp = range_tables[ion]

# Build the precomputed V(y) integral interpolator (speeds up geometry calculations)
F_interp = build_vrint_interpolator(vt_model=vt_model, ion=ion, energy=energy, vb=vb)

# Compute track parameters
result = calculate_track_parameters(
    energy=energy,
    angle_deg=angle,
    vb=vb,
    time_etching=time_etching,
    range_interpolator=range_interp,
    F_interp=F_interp,
    ion=ion,
    vt_model=vt_model,
)

print(f"Track Status: {result['status']}")
print(f"Depth       : {result['depth_um']:.4f} µm")
print(f"Major Axis  : {result['major_axis_um']:.4f} µm")
print(f"Minor Axis  : {result['minor_axis_um']:.4f} µm")
print(f"Black Part %: {result['black_part']*100:.1f}%")

# Plotting the 4-panel diagnostic figure
if result['X_surf'] is not None:
    X, Y, Z = result['X_surf'], result['Y_surf'], result['Z_surf']
    B = result['B_faces'] # Microscope brightness matrix
    
    fig = plt.figure(figsize=(12, 9))
    
    # 3D View
    ax1 = fig.add_subplot(2, 2, 1, projection='3d')
    if B is not None:
        ax1.plot_surface(X, Y, Z, facecolors=cm.gray(B), linewidth=0, antialiased=True)
    ax1.set_title("3D Track Cavity Mesh")
    
    # XY Microscope View
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.set_facecolor('#DCDCDC')
    if B is not None:
        ax2.pcolormesh(X, Y, B, cmap='gray', shading='flat')
    ax2.set_title("Simulated XY Microscope View")
    ax2.set_aspect('equal')
    
    # YZ Profile
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.fill(Y[-1, :], Z[-1, :], alpha=0.15)
    ax3.plot(Y[-1, :], Z[-1, :], lw=2)
    ax3.set_title("YZ Longitudinal Profile")
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect('equal')
    
    # XZ Profile
    ax4 = fig.add_subplot(2, 2, 4)
    ax4.fill(X[-1, :], Z[-1, :], alpha=0.15)
    ax4.plot(X[-1, :], Z[-1, :], lw=2)
    ax4.set_title("XZ Longitudinal Profile")
    ax4.grid(True, alpha=0.3)
    ax4.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

<a id="section4"></a>
## 🗄️ Section 4: Generating and Using Look-Up Tables (LUT)

Generating individual tracks is physically detailed but computationally expensive because of numerical integration and vector ray tracing. To process millions of particles, we precompute a **Look-Up Table (LUT)** in a grid of energy and angles, and then use **fast interpolation** to look up track parameters in milliseconds.

### 🖥️ GUI Usage (Mode 3 & Mode 6)
* **Mode 3 (Reference LUT)**: Go to the **3: Reference Dataset** tab, enter energy/angle ranges, and click **Generate** to export a CSV.
* **Mode 6 (LUT Simulation)**: Go to the **6: LUT Simulation** tab, load the generated CSV, and query track properties instantly or run inverse queries (finding energy/angle from track size).

### 💻 Programmatic Usage
Let's see how to generate a small LUT and query/interpolate values from it using `LUTEngine`:

In [ ]:
from tracklab.lut_engine import LUTEngine

# Let's generate a tiny reference dataset programmatically
print("Generating tiny LUT...")
energies = np.linspace(1.0, 3.0, 5) # 5 energies
angles = np.linspace(60, 90, 4)     # 4 angles

lut_data = []
for energy in energies:
    F_interp = build_vrint_interpolator(vt_model=vt_model, ion='protons', energy=energy, vb=4.7)
    for angle in angles:
        res = calculate_track_parameters(
            energy=energy, angle_deg=angle, vb=4.7, time_etching=3.0,
            range_interpolator=range_tables['protons'], F_interp=F_interp,
            ion='protons', vt_model=vt_model
        )
        lut_data.append({
            'energy_MeV': energy,
            'angle_deg': angle,
            'depth_um': res['depth_um'],
            'major_axis_um': res['major_axis_um'],
            'minor_axis_um': res['minor_axis_um'],
            'total_length_um': res['total_length_um'],
            'status': res['status']
        })

# Save to a temporary CSV
lut_df = pd.DataFrame(lut_data)
csv_path = 'tiny_lut_example.csv'
lut_df.to_csv(csv_path, index=False)
print(f"Saved database with {len(lut_df)} tracks to {csv_path}")

# Load the LUT using TrackLab's LUT Engine for instantaneous interpolation
engine = LUTEngine(csv_path)
print("LUT Engine loaded successfully.")

# Query a value that is NOT in the grid (e.g. Energy=2.2 MeV, Angle=72 degrees)
query_res = engine.query(2.2, 72.0)
print(f"\nInterpolated query (Energy=2.2 MeV, Angle=72°):")
print(f"  Major axis: {query_res['major_axis_um']:.4f} µm")
print(f"  Minor axis: {query_res['minor_axis_um']:.4f} µm")
print(f"  Depth:      {query_res['depth_um']:.4f} µm")

# Clean up
if os.path.exists(csv_path):
    os.remove(csv_path)


<a id="section5"></a>
## 🚀 Section 5: Processing Monte Carlo (FLUKA) Simulations

A major feature of TrackLab is the ability to process full phase-space output files from Monte Carlo codes like FLUKA.

### 🖥️ GUI Usage (Mode 4: FLUKA Process)
1. Navigate to the **4: FLUKA Process** tab.
2. Click **Browse** and select the input file (e.g., `AmBe_01001_Phase-space_BXDRAW.txt`).
3. Click **Process File**.
4. The GUI reads the particle coordinates and direction cosines, computes the entry angle and corrected etching time (reducing etching time if the particle started below the detector surface), and outputs a CSV containing track dimensions for all developed tracks.

### 💻 Programmatic Usage
Let's see how to parse a FLUKA phase-space file programmatically and examine the first few processed tracks:

In [ ]:
from tracklab.mode_use import run_mode4_fluka

# We'll use the provided sample FLUKA file
fluka_file = r"../examples/AmBe_01001_Phase-space_BXDRAW.txt"
output_csv = "fluka_processed_results.csv"

if os.path.exists(fluka_file):
    print("Processing FLUKA file...")
    # Run the processor (we'll process a small subset for demonstration)
    results = run_mode4_fluka(
        ion='protons',
        input_file=fluka_file,
        vb=4.7,
        time_etching=2.83,
        output_csv=output_csv,
        skip_header=1
    )
    
    # Read and show the first 5 results
    df_results = pd.read_csv(output_csv)
    print("\nProcessed Tracks Sample:")
    display(df_results.head(5))
    
    # Plot the major axis distribution of the developed tracks
    developed_only = df_results[df_results['status'] == 'Developed']
    if len(developed_only) > 0:
        plt.figure(figsize=(7, 4.5))
        plt.hist(developed_only['major_axis_um'], bins=15, color='darkorange', edgecolor='black', alpha=0.7)
        plt.xlabel("Major Axis Diameter (µm)")
        plt.ylabel("Count")
        plt.title("Track Size Distribution from AmBe FLUKA Phase-space")
        plt.grid(True, alpha=0.3)
        plt.show()
    
    # Clean up output
    if os.path.exists(output_csv):
        os.remove(output_csv)
else:
    print(f"Sample FLUKA file not found at {fluka_file}. Make sure to run the notebook inside the 'docs' directory.")
